# 02 - Baseline model + MLflow tracking

Cleaning rules come from `01_eda.ipynb`: keep 1-60 min trips with 0 < distance <= 100 miles.
Only features known **when the trip starts** are used (no fares, tips, or dropoff time, to avoid leakage).

In [1]:
import mlflow
import numpy as np
import pandas as pd
from pathlib import Path
from sklearn.dummy import DummyRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.feature_extraction import DictVectorizer
from sklearn.metrics import root_mean_squared_error
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline

ROOT = Path.cwd().parent  # notebook lives in notebooks/
# MLflow 3.x wants a database for run metadata; model files go to ./mlruns
mlflow.set_tracking_uri(f"sqlite:///{(ROOT / 'mlflow.db').as_posix()}")
EXPERIMENT = "nyc-taxi-duration"
if mlflow.get_experiment_by_name(EXPERIMENT) is None:
    mlflow.create_experiment(EXPERIMENT, artifact_location=(ROOT / "mlruns").as_uri())
mlflow.set_experiment(EXPERIMENT)

<Experiment: artifact_location='file:///C:/Users/qutib/Projects/jupyter-notebooks/ML%20Course/mle-model-deployment-project/mlruns', creation_time=1795510486249, effective_trace_archival_retention=None, experiment_id='1', last_update_time=1795510486249, lifecycle_stage='active', name='nyc-taxi-duration', tags={}, trace_location=None, workspace='default'>

In [2]:
# same cleaning as the EDA notebook, now as a function
def load_and_clean(path):
    df = pd.read_parquet(path)
    df["duration"] = (df.tpep_dropoff_datetime - df.tpep_pickup_datetime).dt.total_seconds() / 60
    df = df[(df.duration >= 1) & (df.duration <= 60)]
    df = df[(df.trip_distance > 0) & (df.trip_distance <= 100)]
    return df

df = load_and_clean(ROOT / "data" / "yellow_tripdata_2025-01.parquet")
df.shape

(3339105, 21)

In [3]:
# features: location IDs are categories (strings -> one-hot via DictVectorizer), the rest numeric
CATEGORICAL = ["PULocationID", "DOLocationID"]
NUMERICAL = ["trip_distance", "pickup_hour", "pickup_dow"]

def prepare_features(df):
    out = pd.DataFrame({
        "PULocationID": df.PULocationID.astype(str),
        "DOLocationID": df.DOLocationID.astype(str),
        "trip_distance": df.trip_distance,
        "pickup_hour": df.tpep_pickup_datetime.dt.hour,
        "pickup_dow": df.tpep_pickup_datetime.dt.day_of_week,
    })
    return out.to_dict(orient="records")

# Random Forest on 3M+ rows is slow -> train on a random sample (documented in MLflow)
SAMPLE_SIZE = 200_000
sample = df.sample(SAMPLE_SIZE, random_state=42)
train_df, val_df = train_test_split(sample, test_size=0.2, random_state=42)

X_train, y_train = prepare_features(train_df), train_df.duration.values
X_val, y_val = prepare_features(val_df), val_df.duration.values
len(X_train), len(X_val)

(160000, 40000)

In [4]:
# trivial baseline: always predict the mean -> the number the model must beat
with mlflow.start_run(run_name="mean-baseline"):
    dummy = DummyRegressor(strategy="mean").fit(np.zeros((len(y_train), 1)), y_train)
    rmse = root_mean_squared_error(y_val, dummy.predict(np.zeros((len(y_val), 1))))
    mlflow.log_params({"model": "DummyRegressor(mean)", "sample_size": SAMPLE_SIZE})
    mlflow.log_metric("val_rmse", rmse)
print(f"mean baseline RMSE: {rmse:.2f} min")

mean baseline RMSE: 9.75 min


In [5]:
def train_rf(params, run_name):
    with mlflow.start_run(run_name=run_name):
        pipe = make_pipeline(DictVectorizer(), RandomForestRegressor(random_state=42, n_jobs=-1, **params))
        pipe.fit(X_train, y_train)
        train_rmse = root_mean_squared_error(y_train, pipe.predict(X_train))
        val_rmse = root_mean_squared_error(y_val, pipe.predict(X_val))

        mlflow.log_params({**params, "model": "RandomForestRegressor", "sample_size": SAMPLE_SIZE,
                           "features": ",".join(CATEGORICAL + NUMERICAL),
                           "outlier_rule": "1<=duration<=60, 0<distance<=100"})
        mlflow.log_metrics({"train_rmse": train_rmse, "val_rmse": val_rmse})
        # MLflow saves with skops (safe format); tree internals must be explicitly trusted -
        # fine here because we created the model ourselves
        mlflow.sklearn.log_model(pipe, name="model",
                                 skops_trusted_types=["sklearn.tree._tree.Tree"])
        print(f"{run_name}: train RMSE {train_rmse:.2f} | val RMSE {val_rmse:.2f}")
    return pipe

# start small, then give the forest more capacity
rf_small = train_rf({"n_estimators": 20, "max_depth": 10}, "rf-small")
rf_medium = train_rf({"n_estimators": 50, "max_depth": 20, "min_samples_leaf": 5}, "rf-medium")

2026/11/24 10:48:29 WARNING mlflow.utils.environment: Failed to resolve installed pip version. ``pip`` will be added to conda.yaml environment spec without a version specifier.


rf-small: train RMSE 4.32 | val RMSE 4.51


2026/11/24 10:53:12 WARNING mlflow.utils.environment: Failed to resolve installed pip version. ``pip`` will be added to conda.yaml environment spec without a version specifier.


rf-medium: train RMSE 3.75 | val RMSE 4.30


In [6]:
# which features matter most? (aggregate the one-hot columns back to their source feature)
dv, rf = rf_medium.named_steps["dictvectorizer"], rf_medium.named_steps["randomforestregressor"]
imp = pd.Series(rf.feature_importances_, index=dv.get_feature_names_out())
imp.groupby(lambda c: c.split("=")[0]).sum().sort_values(ascending=False)

trip_distance    0.881768
pickup_hour      0.054409
DOLocationID     0.022757
PULocationID     0.020962
pickup_dow       0.020105
dtype: float64

Compare the runs in the UI from the project root:

```bash
uv run mlflow ui --backend-store-uri sqlite:///mlflow.db --port 5000
```